In this notebook, we will show how to conduct co-simulation for an air-cooled data center with one CRAC unit and a chiller plant. In this case, the control variables are the the CRAC supply air flow rate and the supply air temperature setpoint.

In [ ]:
import numpy as np

from dctwin.interfaces.gym_envs import CoSimEnv
from dctwin.utils import config as env_config
from dctwin.utils import read_engine_config, setup_logging

from google.protobuf import json_format

(1) Setup environment variables

In [ ]:
engine_config = "config.prototxt"
env_config.eplus.engine_config_file = engine_config

(2) Read configuration files

In [ ]:
config = read_engine_config(engine_config=engine_config)
setup_logging(config.logging_config, engine_config=engine_config)

(3) Build environment

In [ ]:
env_config_name = config.WhichOneof("EnvConfig")
env_params = json_format.MessageToDict(
    getattr(config, env_config_name).env_params,
    preserving_proto_field_name=True,
)
env = CoSimEnv(
    config=getattr(config, env_config_name),
    reward_fn=None,
    schedule_fn=None,
    **env_params,
)
env.reset()

(4) Check the name of the CRAC in each AirLoop

In [ ]:
from dctwin.backends.eplus.parser import IDFParser
env.co_sim_manager.eplus_manager.idf_parser = IDFParser(idf_path="model/idf/1ZoneDataCenterHVAC_Alibaba.idf")
for crac in env.co_sim_manager.eplus_manager.idf_parser.epm.AirLoopHVAC:
    uid = env.co_sim_manager.idf2room_mapper[crac.name]
    print(uid)

(5) Specify the action dictionary

In [ ]:
action_dict = {
    "ACU1_setpoint": 20.0,
    "ACU1_flow_rate": 15.0,
    "cpu_loading_schedule": 0.5,
    "chw_supply_sp": 10.0
}

Note: the action_dict should be in the format of: { "{uid}_setpoint": xxx, "{uid}_flow_rate": xxx, "cpu_load_schedule": xxx, ...(other actions for the chiller plant side (e.g., chilled water temperature, ....) to be sent to EnergyPlus) } The "uid" can be obtained by running the previous cell. Each CRAC should have a setpoint and a flow rate. There will be only one "cpu_load_schedule" and the key name must be "cpu_load_schedule".

(6) Run one-step simulation using the specified boundary conditions.

In [ ]:
bc = env.co_sim_manager.map_boundary_conditions(parsed_actions=action_dict)
env.co_sim_manager.cfd_manager.run(**bc)

(7) Run energy simulation with fixed action to obtain energy consumption result Note: we highly recommend you to open the dry run mode to speed up simulation as we use the fixed action to run the simulation and we should not run CFD simulation again! Instead, we use the cached CFD result to pass to EnergyPlus


In [ ]:
act = np.array([
    action_dict["cpu_load_schedule"],
    action_dict["ACU1_setpoint"],
    action_dict["ACU1_flow_rate"],
    action_dict["chw_supply_sp"]
])
env.reset()
done = False
config.cfd.dry_run = True
while not done:
    obs, rew, done, truncated, info = env.step(act)